# Calidad y preparación de datos

Este notebook desarrolla la revisión de calidad y deja los datos listos para el modelamiento del TP1. El escenario principal es una clasificación multiclase (`Dropout`, `Enrolled`, `Graduate`) al cierre del primer semestre. Se preserva el CSV original, no se eliminan filas por valores extremos sin justificación y toda transformación aprendida se ajusta exclusivamente sobre el conjunto de entrenamiento.

La selección temporal sigue el [alcance del proyecto](../README.md): variables de matrícula y contexto y desempeño del 1.er semestre; las seis variables del 2.º semestre se excluyen para evitar *data leakage*. El diccionario está en [reports/diccionario_variables.md](../reports/diccionario_variables.md).

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

DATA_PATH = Path("../data/raw/data.csv")
TARGET = "Target"
EXPECTED_ROWS = 4424
EXPECTED_COLUMNS = 37

raw_df = pd.read_csv(DATA_PATH, sep=";")
raw_df.columns = raw_df.columns.str.strip()

second_semester_cols = [
    col for col in raw_df.columns
    if col.startswith("Curricular units 2nd sem")
]
numeric_df = raw_df.select_dtypes(include=np.number)
count_cols = [
    col for col in raw_df.columns
    if col.startswith("Curricular units") and "(grade)" not in col
]
quality_checks = pd.DataFrame({
    "Control": [
        "Dimensiones esperadas (4,424 x 37)",
        "Columna objetivo presente",
        "Faltantes",
        "Filas duplicadas exactas",
        "Valores infinitos",
        "Valores negativos en conteos curriculares",
        "Columnas del segundo semestre detectadas",
    ],
    "Resultado": [
        raw_df.shape == (EXPECTED_ROWS, EXPECTED_COLUMNS),
        TARGET in raw_df.columns,
        int(raw_df.isna().sum().sum()),
        int(raw_df.duplicated().sum()),
        int(np.isinf(numeric_df.to_numpy()).sum()),
        int(sum((raw_df[col] < 0).sum() for col in count_cols)),
        len(second_semester_cols),
    ],
})

display(quality_checks)
display(raw_df[TARGET].value_counts().rename_axis(TARGET).to_frame("Estudiantes"))

,Control,Resultado
0,"Dimensiones esperadas (4,424 x 37)",True
1,Columna objetivo presente,True
2,Faltantes,0
3,Filas duplicadas exactas,0
4,Valores infinitos,0
5,Valores negativos en conteos curriculares,0
6,Columnas del segundo semestre detectadas,6


,Estudiantes
Target,
Graduate,2209
Dropout,1421
Enrolled,794


## 1. Auditoría inicial

La tabla contrasta dimensiones y controles básicos con la documentación UCI. Un resultado de cero faltantes es una propiedad observada de este archivo, no motivo para omitir un imputador en el pipeline: se incluye para que el flujo siga funcionando ante nuevos datos y se ajusta solo con entrenamiento.

Las filas duplicadas se reportan, pero no se eliminan automáticamente: sin un identificador de estudiante no se puede determinar si dos filas idénticas son errores o registros válidos.

In [12]:
binary_cols = [
    "Daytime/evening attendance", "Displaced", "Educational special needs",
    "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder", "International",
]
nominal_cols = [
    "Marital status", "Application mode", "Course", "Previous qualification",
    "Nacionality", "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
]
ordinal_cols = ["Application order"]

binary_domain_violations = {
    col: sorted(set(raw_df[col].dropna().unique()) - {0, 1})
    for col in binary_cols
    if col in raw_df.columns
}
grade_ranges = {
    "Previous qualification (grade)": (0, 200),
    "Admission grade": (0, 200),
    "Curricular units 1st sem (grade)": (0, 20),
    "Curricular units 2nd sem (grade)": (0, 20),
}
domain_rows = [
    {"Variable": TARGET, "Control": "Clases observadas coinciden con las esperadas", "Resultado": set(raw_df[TARGET].unique()) == {"Dropout", "Enrolled", "Graduate"}},
]
for col, invalid_values in binary_domain_violations.items():
    domain_rows.append({
        "Variable": col,
        "Control": "Valores fuera del dominio binario {0, 1}",
        "Resultado": invalid_values,
    })
for col, (lower, upper) in grade_ranges.items():
    if col in raw_df.columns:
        invalid_count = int((~raw_df[col].between(lower, upper)).sum())
        domain_rows.append({
            "Variable": col,
            "Control": f"Valores fuera del rango [{lower}, {upper}]",
            "Resultado": invalid_count,
        })
display(pd.DataFrame(domain_rows))

rare_category_rows = []
for col in nominal_cols:
    counts = raw_df[col].value_counts(dropna=False)
    rare_category_rows.append({
        "Variable": col,
        "Categorías": int(counts.size),
        "Categorías con menos de 30 casos": int((counts < 30).sum()),
        "Estudiantes en categorías < 30": int(counts[counts < 30].sum()),
    })
display(pd.DataFrame(rare_category_rows).set_index("Variable"))

first_semester_count_cols = [
    col for col in raw_df.columns
    if col.startswith("Curricular units 1st sem") and col != "Curricular units 1st sem (grade)"
]
zero_by_target = raw_df.groupby(TARGET)[first_semester_count_cols].agg(lambda values: int(values.eq(0).sum())).T
zero_by_target.columns.name = "Target: cantidad de ceros"
display(zero_by_target)

,Variable,Control,Resultado
0,Target,Clases observadas coinciden con las esperadas,True
1,Daytime/evening attendance,"Valores fuera del dominio binario {0, 1}",[]
2,Displaced,"Valores fuera del dominio binario {0, 1}",[]
3,Educational special needs,"Valores fuera del dominio binario {0, 1}",[]
4,Debtor,"Valores fuera del dominio binario {0, 1}",[]
5,Tuition fees up to date,"Valores fuera del dominio binario {0, 1}",[]
6,Gender,"Valores fuera del dominio binario {0, 1}",[]
7,Scholarship holder,"Valores fuera del dominio binario {0, 1}",[]
8,International,"Valores fuera del dominio binario {0, 1}",[]
9,Previous qualification (grade),"Valores fuera del rango [0, 200]",0


,Categorías,Categorías con menos de 30 casos,Estudiantes en categorías < 30
Variable,,,
Marital status,6,3,35
Application mode,18,6,32
Course,17,1,12
Previous qualification,17,10,79
Nacionality,21,19,72
Mother's qualification,29,20,89
Father's qualification,34,25,102
Mother's occupation,32,21,116
Father's occupation,46,34,112


Target: cantidad de ceros,Dropout,Enrolled,Graduate
Curricular units 1st sem (credited),1242,697,1908
Curricular units 1st sem (enrolled),77,28,75
Curricular units 1st sem (evaluations),246,28,75
Curricular units 1st sem (approved),570,71,77
Curricular units 1st sem (without evaluations),1297,724,2109


### Revisiones de consistencia y valores extremos

Se verifican relaciones lógicas entre conteos académicos y se resumen posibles valores extremos con la regla de 1,5 rangos intercuartílicos (IQR). Los puntos marcados son candidatos a revisión, no errores automáticos: por la limpieza previa documentada por UCI y la ausencia de información del caso, se conservan y no se recortan ni eliminan sin justificación.

In [13]:
consistency_rows = []
for semester in ["1st", "2nd"]:
    enrolled_col = f"Curricular units {semester} sem (enrolled)"
    approved_col = f"Curricular units {semester} sem (approved)"
    without_eval_col = f"Curricular units {semester} sem (without evaluations)"
    if enrolled_col in raw_df.columns and approved_col in raw_df.columns:
        consistency_rows.append({
            "Regla revisada": f"Aprobadas <= matriculadas ({semester} semestre)",
            "Casos señalados": int((raw_df[approved_col] > raw_df[enrolled_col]).sum()),
        })
    if enrolled_col in raw_df.columns and without_eval_col in raw_df.columns:
        consistency_rows.append({
            "Regla revisada": f"Sin evaluaciones > matriculadas ({semester} semestre)",
            "Casos señalados": int((raw_df[without_eval_col] > raw_df[enrolled_col]).sum()),
        })
display(pd.DataFrame(consistency_rows))

continuous_features = [
    "Age at enrollment", "Previous qualification (grade)", "Admission grade",
    "Curricular units 1st sem (grade)", "Unemployment rate", "Inflation rate", "GDP",
]
iqr_rows = []
for col in [name for name in continuous_features if name in raw_df.columns]:
    q1 = raw_df[col].quantile(0.25)
    q3 = raw_df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    below = int((raw_df[col] < lower_bound).sum())
    above = int((raw_df[col] > upper_bound).sum())
    iqr_rows.append({
        "Variable continua": col,
        "Límite inferior IQR": round(lower_bound, 2),
        "Límite superior IQR": round(upper_bound, 2),
        "Casos bajo el límite": below,
        "Casos sobre el límite": above,
        "Casos señalados": below + above,
    })
display(pd.DataFrame(iqr_rows).set_index("Variable continua"))

curricular_count_cols = [
    col for col in count_cols
    if col.startswith("Curricular units 1st sem")
]
count_distribution = raw_df[curricular_count_cols].describe(percentiles=[0.5, 0.95]).T[
    ["min", "50%", "95%", "max"]
]
display(count_distribution.rename_axis("Conteo curricular").round(1))

,Regla revisada,Casos señalados
0,Aprobadas <= matriculadas (1st semestre),0
1,Sin evaluaciones > matriculadas (1st semestre),0
2,Aprobadas <= matriculadas (2nd semestre),0
3,Sin evaluaciones > matriculadas (2nd semestre),3


,Límite inferior IQR,Límite superior IQR,Casos bajo el límite,Casos sobre el límite,Casos señalados
Variable continua,,,,,
Age at enrollment,10.00,34.00,0,441,441
Previous qualification (grade),102.50,162.50,93,86,179
Admission grade,92.55,160.15,0,86,86
Curricular units 1st sem (grade),7.40,17.00,718,8,726
Unemployment rate,2.65,20.65,0,0,0
Inflation rate,-3.15,6.05,0,0,0
GDP,-6.94,7.02,0,0,0


,min,50%,95%,max
Conteo curricular,,,,
Curricular units 1st sem (credited),0.0,0.0,6.0,20.0
Curricular units 1st sem (enrolled),0.0,6.0,11.0,26.0
Curricular units 1st sem (evaluations),0.0,8.0,15.0,45.0
Curricular units 1st sem (approved),0.0,5.0,9.0,26.0
Curricular units 1st sem (without evaluations),0.0,0.0,1.0,12.0


**Lectura de la auditoría:** no hay violaciones en las banderas binarias ni en los rangos publicados para notas. Hay 3 registros donde las unidades “sin evaluaciones” del segundo semestre superan las matriculadas; quedan señalados, sin corrección automática. La regla IQR marca edades por encima de 34 años y parte de las notas bajas, pero son observaciones dentro de sus dominios válidos; no se recortan ni eliminan sin evidencia adicional. Los conteos curriculares se inspeccionan aparte porque los ceros y la distribución discreta hacen poco informativa una regla IQR genérica.

## 2. Reglas de preparación

- **Objetivo:** conservar las tres clases; la alternativa binaria `Dropout` vs. no `Dropout` se podrá derivar después para comparar formulaciones.
- **Disponibilidad:** excluir las seis columnas del segundo semestre. Las variables macroeconómicas se conservan según el alcance del proyecto, pero su fecha de referencia debe verificarse antes de una alerta operativa.
- **Categorías:** codificar las nominales con one-hot; mantener las binarias 0/1 y tratar `Application order` como ordinal, no como nominal ni como un código continuo indiferenciado.
- **Calidad:** no eliminar observaciones ni convertir ceros en nulos sin evidencia del diccionario. Se incluyen imputadores como parte del pipeline para tratar datos nuevos si presentan faltantes.
- **Sin fuga:** dividir primero en entrenamiento y prueba; ajustar imputación, escalado y categorías solo con entrenamiento. No aplicar sobremuestreo ni selección supervisada en este notebook.

In [14]:
from sklearn.model_selection import train_test_split

X_all = raw_df.drop(columns=TARGET)
y = raw_df[TARGET].copy()
X = X_all.drop(columns=second_semester_cols)

categorical_features = [col for col in nominal_cols if col in X.columns]
binary_features = [col for col in binary_cols if col in X.columns]
ordinal_features = [col for col in ordinal_cols if col in X.columns]
numeric_features = [
    col for col in X.columns
    if col not in categorical_features + binary_features + ordinal_features
]

feature_groups = pd.DataFrame({
    "Grupo semántico": ["Categóricas nominales", "Binarias", "Ordinales", "Cuantitativas", "Total"],
    "Número de variables": [
        len(categorical_features), len(binary_features), len(ordinal_features),
        len(numeric_features), X.shape[1],
    ],
    "Variables": [
        categorical_features, binary_features, ordinal_features, numeric_features,
        X.columns.tolist(),
    ],
})
display(feature_groups)

feature_exclusion = pd.DataFrame({
    "Grupo excluido": ["Variables del 2.º semestre", "Objetivo"],
    "Número de columnas": [len(second_semester_cols), 1],
    "Columnas": [second_semester_cols, [TARGET]],
})
display(feature_exclusion)

assert TARGET not in X.columns
assert not set(second_semester_cols).intersection(X.columns)
assert X.shape[1] == 30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

split_summary = pd.concat(
    {
        "Conjunto completo": y.value_counts(normalize=True),
        "Entrenamiento": y_train.value_counts(normalize=True),
        "Prueba": y_test.value_counts(normalize=True),
    },
    axis=1,
).fillna(0).mul(100).rename_axis("Target")
display(split_summary.round(2))
display(pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Prueba"],
    "Estudiantes": [len(X_train), len(X_test)],
    "Predictores": [X_train.shape[1], X_test.shape[1]],
}))

,Grupo semántico,Número de variables,Variables
0,Categóricas nominales,9,"[Marital status, Application mode, Course, Pre..."
1,Binarias,8,"[Daytime/evening attendance, Displaced, Educat..."
2,Ordinales,1,[Application order]
3,Cuantitativas,12,"[Previous qualification (grade), Admission gra..."
4,Total,30,"[Marital status, Application mode, Application..."


,Grupo excluido,Número de columnas,Columnas
0,Variables del 2.º semestre,6,"[Curricular units 2nd sem (credited), Curricul..."
1,Objetivo,1,[Target]


,Conjunto completo,Entrenamiento,Prueba
Target,,,
Graduate,49.93,49.93,49.94
Dropout,32.12,32.13,32.09
Enrolled,17.95,17.94,17.97


,Conjunto,Estudiantes,Predictores
0,Entrenamiento,3539,30
1,Prueba,885,30


## 3. Pipeline de transformación sin fuga

El preprocesamiento se integra en un `ColumnTransformer` para que el mismo ajuste se reutilice en validación y producción. Las categorías nominales reciben imputación por moda y one-hot; las binarias conservan su codificación 0/1; `Application order` y las variables cuantitativas reciben imputación por mediana y estandarización. Los faltantes no se inventan en el dataset actual: los imputadores son parte del flujo para datos futuros.

`OneHotEncoder(handle_unknown="ignore")` permite transformar valores categóricos del conjunto de prueba que no aparecieron en entrenamiento, algo relevante para las categorías de baja frecuencia observadas en la auditoría.

In [15]:
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])
scaled_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("nominal", nominal_pipeline, categorical_features),
        ("binary", binary_pipeline, binary_features),
        ("ordinal", scaled_pipeline, ordinal_features),
        ("numeric", scaled_pipeline, numeric_features),
    ],
    remainder="drop",
    sparse_threshold=1.0,
    verbose_feature_names_out=True,
)

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)
prepared_feature_names = preprocessor.get_feature_names_out()

if sparse.issparse(X_train_prepared):
    finite_train_values = bool(np.isfinite(X_train_prepared.data).all())
    finite_test_values = bool(np.isfinite(X_test_prepared.data).all())
else:
    finite_train_values = bool(np.isfinite(X_train_prepared).all())
    finite_test_values = bool(np.isfinite(X_test_prepared).all())

preparation_summary = pd.DataFrame({
    "Métrica": [
        "Predictores de entrada",
        "Características después de codificar",
        "Estudiantes en entrenamiento",
        "Estudiantes en prueba",
        "Salida dispersa (sparse)",
        "Valores finitos en entrenamiento",
        "Valores finitos en prueba",
        "Columnas del 2.º semestre en entrada",
    ],
    "Valor": [
        X_train.shape[1],
        X_train_prepared.shape[1],
        X_train_prepared.shape[0],
        X_test_prepared.shape[0],
        sparse.issparse(X_train_prepared),
        finite_train_values,
        finite_test_values,
        len(set(second_semester_cols).intersection(X.columns)),
    ],
})
display(preparation_summary)
display(pd.DataFrame({"Característica transformada": prepared_feature_names[:30]}))

unseen_category_rows = []
for col in categorical_features:
    train_levels = set(X_train[col].dropna().unique())
    test_levels = set(X_test[col].dropna().unique())
    unseen = sorted(test_levels - train_levels)
    unseen_category_rows.append({
        "Variable nominal": col,
        "Niveles en prueba no vistos en entrenamiento": len(unseen),
        "Códigos": unseen,
    })
display(pd.DataFrame(unseen_category_rows).set_index("Variable nominal"))

assert TARGET not in X.columns
assert not set(second_semester_cols).intersection(X.columns)
assert X_train_prepared.shape[1] == len(prepared_feature_names)
assert finite_train_values and finite_test_values
assert len(y_train) == X_train_prepared.shape[0]
assert len(y_test) == X_test_prepared.shape[0]

,Métrica,Valor
0,Predictores de entrada,30
1,Características después de codificar,230
2,Estudiantes en entrenamiento,3539
3,Estudiantes en prueba,885
4,Salida dispersa (sparse),True
5,Valores finitos en entrenamiento,True
6,Valores finitos en prueba,True
7,Columnas del 2.º semestre en entrada,0


,Característica transformada
0,nominal__Marital status_1
1,nominal__Marital status_2
2,nominal__Marital status_3
3,nominal__Marital status_4
4,nominal__Marital status_5
5,nominal__Marital status_6
6,nominal__Application mode_1
7,nominal__Application mode_2
8,nominal__Application mode_5
9,nominal__Application mode_7


,Niveles en prueba no vistos en entrenamiento,Códigos
Variable nominal,,
Marital status,0,[]
Application mode,0,[]
Course,0,[]
Previous qualification,0,[]
Nacionality,2,"[17, 109]"
Mother's qualification,1,[27]
Father's qualification,2,"[13, 33]"
Mother's occupation,1,[125]
Father's occupation,5,"[114, 121, 124, 132, 174]"


## 4. Salidas para el modelamiento del TP1

`X_train_prepared` y `X_test_prepared` son las matrices transformadas; `y_train` y `y_test` conservan las tres clases para el escenario principal. Se crea además una versión binaria auxiliar para la comparación del TP1. La división de prueba queda separada y no debe usarse para elegir transformaciones, hiperparámetros ni umbrales; esas decisiones deben realizarse con validación dentro de entrenamiento.

No se balancean clases aquí. El baseline y los modelos se evaluarán en el siguiente paso con la distribución natural y las métricas definidas en el README (recall de `Dropout`, F1 macro y accuracy de referencia).

In [16]:
y_train_binary = y_train.map(lambda value: "Dropout" if value == "Dropout" else "No Dropout")
y_test_binary = y_test.map(lambda value: "Dropout" if value == "Dropout" else "No Dropout")

binary_target_summary = pd.concat(
    {
        "Entrenamiento": y_train_binary.value_counts(normalize=True),
        "Prueba": y_test_binary.value_counts(normalize=True),
    },
    axis=1,
).fillna(0).mul(100).rename_axis("Target binario")
display(binary_target_summary.round(2))

data_handoff = pd.DataFrame({
    "Objeto": [
        "X_train_prepared", "X_test_prepared", "y_train", "y_test",
        "y_train_binary", "y_test_binary", "preprocessor",
    ],
    "Contenido": [
        f"{X_train_prepared.shape[0]} x {X_train_prepared.shape[1]} transformada",
        f"{X_test_prepared.shape[0]} x {X_test_prepared.shape[1]} transformada",
        f"{len(y_train)} etiquetas multiclase",
        f"{len(y_test)} etiquetas multiclase",
        f"{len(y_train_binary)} etiquetas binarias",
        f"{len(y_test_binary)} etiquetas binarias",
        "Ajustado solo con X_train",
    ],
})
display(data_handoff)

assert set(y_train.unique()) == {"Dropout", "Enrolled", "Graduate"}
assert set(y_test.unique()) == {"Dropout", "Enrolled", "Graduate"}
assert set(y_train_binary.unique()) == {"Dropout", "No Dropout"}
assert set(y_test_binary.unique()) == {"Dropout", "No Dropout"}
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_train.index).union(X_test.index) == set(X.index)

,Entrenamiento,Prueba
Target binario,,
No Dropout,67.87,67.91
Dropout,32.13,32.09


,Objeto,Contenido
0,X_train_prepared,3539 x 230 transformada
1,X_test_prepared,885 x 230 transformada
2,y_train,3539 etiquetas multiclase
3,y_test,885 etiquetas multiclase
4,y_train_binary,3539 etiquetas binarias
5,y_test_binary,885 etiquetas binarias
6,preprocessor,Ajustado solo con X_train
